# Run the VGGT reconstruction on a free Colab GPU

For machines without a capable GPU. This runs the same frame selection and VGGT step as `run.py`, then you download the result and finish locally:

```
uv run python run.py BEFORE_VIDEO AFTER_VIDEO --reconstruction vggt_30f.npz
```

1. Runtime > Change runtime type > **T4 GPU**
2. Runtime > Run all
3. Upload the two videos when asked (before first, then after); the result downloads at the end

In [ ]:
REPO_URL = "https://github.com/kirill271204/RoomDelta.git"
FRAMES_PER_VIDEO = 30   # a 15 GB T4 handles 30 per video (60 total)

!test -d /content/RoomDelta || git clone -q $REPO_URL /content/RoomDelta
!test -d /content/RoomDelta/third_party/vggt || git clone -q https://github.com/facebookresearch/vggt.git /content/RoomDelta/third_party/vggt
# Only VGGT's small extras: installing the repo itself would replace Colab's torch and numpy
!pip install -q einops safetensors huggingface_hub
import sys
sys.path.insert(0, '/content/RoomDelta/src')

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU: set Runtime > Change runtime type > T4 GPU'
from pathlib import Path
from google.colab import files

print('Upload the BEFORE video'); before = Path(next(iter(files.upload())))
print('Upload the AFTER video');  after = Path(next(iter(files.upload())))

In [ ]:
from room_change.frames import extract_frames, video_name
from room_change.reconstruct import run_vggt

names = [video_name(before), video_name(after)]
if names[0] == names[1]:
    names = [names[0] + '_before', names[1] + '_after']  # same rule as run.py
for video, name in zip((before, after), names):
    extract_frames(video, Path('frames') / name, FRAMES_PER_VIDEO)
out = Path(f'vggt_{FRAMES_PER_VIDEO}f.npz')
run_vggt([Path('frames') / n for n in names], names, FRAMES_PER_VIDEO, out)
print('peak GPU memory: %.1f GB' % (torch.cuda.max_memory_allocated() / 1e9))
files.download(str(out))